In [40]:
import os
import torch
from collections import Counter

# 设置路径
base_path = "/home/cz/consel/MolTC-main/data/ddi_data/ChChMiner_3d_20_deep/valid/target/"

label_counter = Counter()

# 遍历 0, 1, 2, ... 子目录
for subdir in os.listdir(base_path):
    subdir_path = os.path.join(base_path, subdir)
    if os.path.isdir(subdir_path):
        text_path = os.path.join(subdir_path, "text.pt")
        if os.path.exists(text_path):
            try:
                tensor = torch.load(text_path)
                if isinstance(tensor, torch.Tensor):
                    # 将所有值 flatten 成一维，统计 0/1
                    flat_tensor = tensor.view(-1).tolist()
                    label_counter.update(flat_tensor)
                else:
                    print(f"Warning: {text_path} does not contain a tensor.")
            except Exception as e:
                print(f"Failed to load {text_path}: {e}")
        else:
            print(f"Missing text.pt in {subdir_path}")

# 输出结果
total = sum(label_counter.values())
print("\nClass distribution:")
for label, count in label_counter.items():
    percent = 100.0 * count / total if total > 0 else 0
    print(f"Label {int(label)}: {count} samples ({percent:.2f}%)")

# 简单类别不均衡判断
if len(label_counter) == 2:
    imbalance_ratio = max(label_counter.values()) / min(label_counter.values())
    if imbalance_ratio > 1.5:
        print("\n⚠️ Warning: Class imbalance detected (ratio > 1.5).")
    else:
        print("\n✅ Class distribution looks balanced.")
else:
    print("\n⚠️ Unexpected number of classes detected.")


/tmp/ipykernel_3784724/2428031415.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tensor = torch.load(text_path)



Class distribution:
Label 1: 943 samples (50.00%)
Label 0: 943 samples (50.00%)

✅ Class distribution looks balanced.


In [35]:
import pandas as pd
from collections import Counter

# CSV 文件路径
csv_path = "/home/cz/consel/MolTC-main/data/ddi_data/ChChMiner_valid.csv"

# 读取 CSV
df = pd.read_csv(csv_path)

# 检查是否包含 label 列
if 'label' not in df.columns:
    raise ValueError("❌ CSV 文件中未找到 'label' 列。请确认列名正确。")

# 统计类别分布
label_counts = Counter(df['label'])
total = sum(label_counts.values())

print("\n✅ 类别分布：")
for label, count in sorted(label_counts.items()):
    percent = 100.0 * count / total
    print(f"标签 {label}: {count} 个样本 ({percent:.2f}%)")

# 判断类别是否不均衡
if len(label_counts) == 2:
    imbalance_ratio = max(label_counts.values()) / min(label_counts.values())
    if imbalance_ratio > 1.5:
        print("\n⚠️ 类别不均衡（比例 > 1.5），建议使用采样、加权损失等方法处理。")
    else:
        print("\n✅ 类别分布相对均衡。")
else:
    print("\n⚠️ 类别数量异常（非二分类），请检查数据。")



✅ 类别分布：
标签 0.0: 943 个样本 (18.41%)
标签 1.0: 4178 个样本 (81.59%)

⚠️ 类别不均衡（比例 > 1.5），建议使用采样、加权损失等方法处理。


In [36]:
import pandas as pd

# 原始 CSV 路径
input_path = "/home/cz/consel/MolTC-main/data/ddi_data/ChChMiner_valid.csv"
# 输出路径（避免覆盖）
output_path = "/home/cz/consel/MolTC-main/data/ddi_data/ChChMiner_valid_balanced.csv"

# 读取原始数据
df = pd.read_csv(input_path)

# 按 label 分组
df_label_0 = df[df['label'] == 0]
df_label_1 = df[df['label'] == 1]

# 保留所有 label=0，随机采样与其数量相同的 label=1
df_label_1_sampled = df_label_1.sample(n=len(df_label_0), random_state=42)

# 合并并打乱顺序
df_balanced = pd.concat([df_label_0, df_label_1_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

# 保存为新的 CSV 文件
df_balanced.to_csv(output_path, index=False)

print(f"✅ 已生成均衡数据文件，共 {len(df_balanced)} 行，保存在：\n{output_path}")


✅ 已生成均衡数据文件，共 1886 行，保存在：
/home/cz/consel/MolTC-main/data/ddi_data/ChChMiner_valid_balanced.csv


In [1]:
import os
import re

def process_text_files(root_folder):
    for folder in os.listdir(root_folder):
        folder_path = os.path.join(root_folder, folder)
        if os.path.isdir(folder_path):  # 确保是文件夹
            for file in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file)
                if os.path.isfile(file_path) and file.endswith(".txt"):  # 确保是文本文件
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                    
                    # 移除 <start_property> 和 <end_property>
                    content = re.sub(r'<start_property>|<end_property>', '', content)
                    
                    # 在结尾添加 `.python`
                    content = content.strip() + '.'
                    
                    with open(file_path, 'w', encoding='utf-8') as f:
                        f.write(content)

if __name__ == "__main__":
    root_folder = "/work/home/ac15y5ara4/cz/MolTC-main/data/ddi_data/Zhangddi_data_3d/train/text"
    process_text_files(root_folder)
    print("所有文本文件已修改。")


所有文本文件已修改。


In [2]:
import os
import re
from concurrent.futures import ProcessPoolExecutor

def process_file(file_path):
    if os.path.isfile(file_path) and file_path.endswith(".txt"):  # 确保是文本文件
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # 移除 <start_property> 和 <end_property>
        content = re.sub(r'<start_property>|<end_property>', '', content)
        
        # 在结尾添加 `.`
        content = content.strip() + '.'
        
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)

def process_text_files(root_folder):
    file_paths = []
    for folder in os.listdir(root_folder):
        folder_path = os.path.join(root_folder, folder)
        if os.path.isdir(folder_path):  # 确保是文件夹
            for file in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file)
                file_paths.append(file_path)
    
    # 使用多进程处理文件
    with ProcessPoolExecutor() as executor:
        executor.map(process_file, file_paths)

if __name__ == "__main__":
    root_folder = "/work/home/ac15y5ara4/cz/MolTC-main/data/ddi_data/ChChMiner_3d_20_test/valid/text"
    process_text_files(root_folder)
    print("所有文本文件已修改。")

所有文本文件已修改。


In [2]:
import os
import torch

# 原始txt文件所在目录
base_text_dir = "/home/cz/consel/MolTC-main/data/data/cz/ccgnet-main/data/mydata/gongjin/valid/text"
# 目标保存目录
base_save_dir = "/home/cz/consel/MolTC-main/data/data/cz/ccgnet-main/data/mydata/gongjin/valid/target"

# 确保目标目录存在
os.makedirs(base_save_dir, exist_ok=True)

# 遍历所有数字文件夹
for folder_name in os.listdir(base_text_dir):
    folder_path = os.path.join(base_text_dir, folder_name)
    
    # 确保是一个文件夹
    if not os.path.isdir(folder_path):
        continue

    # 在目标目录创建对应的数字文件夹
    save_folder = os.path.join(base_save_dir, folder_name)
    os.makedirs(save_folder, exist_ok=True)

    # 遍历该文件夹下的所有txt文件
    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".txt"):
            continue
        
        file_path = os.path.join(folder_path, file_name)
        
        # 读取txt文件内容
        with open(file_path, "r") as f:
            content = f.read().strip()
        
        # 提取第一个数字（假设格式是"The answer is 111." 或 "The answer is 000."）
        label = int(content.split()[-1][0])  # 取最后一个单词的第一个字符
        # print(label)
        # 转换为PyTorch tensor
        tensor = torch.tensor(label, dtype=torch.float32)

        # 保存tensor到.pt文件
        tensor_save_path = os.path.join(save_folder, file_name.replace(".txt", ".pt"))
        torch.save(tensor, tensor_save_path)

print("所有txt文件已处理并转换为tensor格式保存。")


所有txt文件已处理并转换为tensor格式保存。


In [15]:
import torch
a=torch.load('/work/home/ac15y5ara4/cz/MolTC-main/data/ddi_data/ChChMiner_3d_20_test/train/target/5/text.pt')
a

/tmp/ipykernel_16187/1568840266.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  a=torch.load('/work/home/ac15y5ara4/cz/MolTC-main/data/ddi_data/ChChMiner_3d_20_test/trai

tensor(1.)

In [6]:
a+1

tensor(2.)